# Module 6: Reading and Writing Data

Spark is a **compute engine** — it reads data from storage, processes it, and writes results back. Choosing the right storage format is one of the most impactful decisions in Big Data:

| Format | Type | Schema | Compression | Best For |
|--------|------|--------|-------------|----------|
| **CSV** | Text, row-based | None (must infer) | None | Human-readable exchange |
| **JSON** | Text, row-based | Self-describing | None | APIs, semi-structured data |
| **Parquet** | Binary, columnar | Embedded | Built-in (snappy) | Analytics — Spark's preferred format |

**Always use Parquet** for Spark analytics. It's smaller, faster, and self-describing.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Module 06 - Reading Writing Data") \
    .master("local[*]") \
    .getOrCreate()

employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

---
## Concept 1: CSV Reading with Options

CSV reading supports many options for handling messy real-world files: delimiters, date formats, null markers, encoding, etc.

In [ ]:
# Verbose form showing common options
sales_csv = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .option("nullValue", "NA")
    .csv("../data/sales.csv")
)
sales_csv.show(5)

#### Try It: Read with Options

Read `../data/employees.csv` using the verbose `.option()` form with:
- `header` = true
- `inferSchema` = true
- `nullValue` = "N/A"

Show the first 5 rows and print schema.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
emp_csv = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("nullValue", "N/A")
    .csv("../data/employees.csv")
)
emp_csv.show(5)
emp_csv.printSchema()

---
## Concept 2: Writing and Reading Parquet

Parquet is a **columnar** format — instead of storing row by row, it stores column by column. This means:
- If you only need 2 columns from a 100-column table, Spark reads only those 2 columns (column pruning)
- Compression is much better because similar values are stored together
- Schema is embedded — no need for `inferSchema`

**In production Big Data, Parquet is the default format.** Period.

In [ ]:
# Write sales as Parquet
sales.write.mode("overwrite").parquet("../output/sales_parquet")
print("Written!")

In [ ]:
# Read it back — schema is preserved automatically (no inferSchema needed!)
sales_from_parquet = spark.read.parquet("../output/sales_parquet")
sales_from_parquet.printSchema()
sales_from_parquet.show(5)

#### Try It: Write and Read Parquet

1. Write the `employees` DataFrame as Parquet to `../output/employees_parquet`
2. Read it back into a new variable
3. Verify the schema is preserved with `.printSchema()`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
employees.write.mode("overwrite").parquet("../output/employees_parquet")
emp_back = spark.read.parquet("../output/employees_parquet")
emp_back.printSchema()
emp_back.show(5)

---
## Concept 3: Writing and Reading JSON

JSON is self-describing (column names are in every row), but verbose and slow. Use it when exchanging data with web APIs or when humans need to read the output.

In [ ]:
# Write and read JSON
employees.write.mode("overwrite").json("../output/employees_json")
emp_from_json = spark.read.json("../output/employees_json")
emp_from_json.show(5)

#### Try It: Write Departments as JSON

1. Write `departments` as JSON to `../output/departments_json`
2. Read it back
3. Show all rows and print schema

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
departments.write.mode("overwrite").json("../output/departments_json")
dept_back = spark.read.json("../output/departments_json")
dept_back.show()
dept_back.printSchema()

---
## Concept 4: Partitioned Writes

Partitioning writes data into **subdirectories** organized by column values. This is a critical optimization:

If you partition by `region` and later query `WHERE region = 'East'`, Spark reads ONLY the `region=East` directory — skipping 75% of the data.

**In production**: partition by columns you frequently filter on (date, region, country, etc.).

In [ ]:
# Write partitioned by region
sales.write.mode("overwrite").partitionBy("region").parquet("../output/sales_by_region")
print("Written with partitions!")

In [ ]:
# See the partition directory structure
import os
for item in sorted(os.listdir("../output/sales_by_region")):
    if not item.startswith(".") and not item.startswith("_"):
        print(item)

In [ ]:
# Read a single partition — only the East data
east_sales = spark.read.parquet("../output/sales_by_region/region=East")
print(f"East region sales: {east_sales.count()}")
east_sales.show(5)

#### Try It: Partition Writes

1. Write sales partitioned by `region` as Parquet to `../output/sales_partitioned`
2. List the partition directories using `os.listdir()`
3. Read back just the `Central` partition and count the rows

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
sales.write.mode("overwrite").partitionBy("region").parquet("../output/sales_partitioned")

for item in sorted(os.listdir("../output/sales_partitioned")):
    if item.startswith("region="):
        print(item)

central = spark.read.parquet("../output/sales_partitioned/region=Central")
print(f"\nCentral sales: {central.count()} rows")

---
## Concept 5: File Size Comparison

Let's prove that Parquet is smaller. We'll write the same data in all 3 formats and compare sizes.

In [ ]:
def dir_size(path):
    """Total size of all files in a directory."""
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total

# Write in all formats
sales.write.mode("overwrite").csv("../output/sales_csv", header=True)
sales.write.mode("overwrite").json("../output/sales_json")
# Parquet already written above

csv_size = dir_size("../output/sales_csv")
json_size = dir_size("../output/sales_json")
parquet_size = dir_size("../output/sales_parquet")

print(f"CSV:     {csv_size:>8,} bytes")
print(f"JSON:    {json_size:>8,} bytes")
print(f"Parquet: {parquet_size:>8,} bytes")
print(f"\nParquet is {csv_size / parquet_size:.1f}x smaller than CSV")

#### Try It: Compare Employee File Sizes

Write the `employees` DataFrame in CSV, JSON, and Parquet formats. Compare sizes. Which is smallest?

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
employees.write.mode("overwrite").csv("../output/emp_csv", header=True)
employees.write.mode("overwrite").json("../output/emp_json")
employees.write.mode("overwrite").parquet("../output/emp_parquet")

for name, path in [("CSV", "../output/emp_csv"), ("JSON", "../output/emp_json"), ("Parquet", "../output/emp_parquet")]:
    print(f"{name:>8}: {dir_size(path):>8,} bytes")

---
## Write Modes

| Mode | Behavior |
|------|----------|
| `overwrite` | Replace existing data |
| `append` | Add to existing data |
| `ignore` | Do nothing if data exists |
| `error` (default) | Throw an error if data exists |

---
## Capstone Exercise

1. Join employees + departments
2. Write the joined result as Parquet, **partitioned by `location`**
3. List the partition directories
4. Read back only the **San Francisco** partition
5. Show the names and salaries of San Francisco employees

In [ ]:
# Your capstone code here


In [ ]:
# --- Capstone Solution ---
from pyspark.sql.functions import col

emp_dept = employees.join(departments, on="department_id")
emp_dept.write.mode("overwrite").partitionBy("location").parquet("../output/emp_dept_by_location")

print("Partitions:")
for item in sorted(os.listdir("../output/emp_dept_by_location")):
    if item.startswith("location="):
        print(f"  {item}")

sf = spark.read.parquet("../output/emp_dept_by_location/location=San Francisco")
print(f"\nSan Francisco employees: {sf.count()}")
sf.select("name", "department_name", "salary").orderBy(col("salary").desc()).show()

In [ ]:
spark.stop()

---
## What You Learned

- **CSV** options handle messy files (header, inferSchema, nullValue, dateFormat)
- **Parquet** is Spark's preferred format — columnar, compressed, schema-embedded
- **JSON** is self-describing but verbose — use for API interchange
- **Partitioned writes** (`partitionBy`) organize data into subdirectories for faster filtered reads
- **Write modes** control behavior when output already exists
- Parquet files are significantly smaller than CSV or JSON

Next: **Module 7 — UDFs and Complex Types**, where you'll write custom functions and work with nested data.